## Installation

In [1]:
!pip install gradio
!pip install transformers

## Image Classification

In [2]:
import gradio as gr
import torch
import torchvision.models as models
from torchvision import transforms
import requests
from PIL import Image

alexnet = models.alexnet(weights='DEFAULT').eval()
raw_labels = requests.get("https://gist.githubusercontent.com/yrevar/942d3a0ac09ec9e5eb3a/raw/238f720ff059c1f82f368259d1ca4ffa5dd8f9f5/imagenet1000_clsidx_to_labels.txt").text
labels = [i.split(',')[0] for i in list(eval(raw_labels).values())]

def inference(data):
    image = transforms.ToTensor()(Image.fromarray(data.astype('uint8'), 'RGB')).unsqueeze(0)
    prediction = torch.nn.functional.softmax(alexnet(image)[0], dim=0)
    dictionary = dict(zip(labels, map(float, prediction)))
    return dictionary

gr.Interface(fn=inference,
             inputs=gr.Image(),
             outputs=gr.Label(num_top_classes=5)).launch(share=True) #, debug=True Use in Colab

Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:06<00:00, 36.9MB/s]


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8e8d93d30fa77ba525.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Text Generation

In [3]:
import tensorflow as tf
from transformers import TFGPT2LMHeadModel, GPT2Tokenizer

# Loading and setting up GPT2 Open AI Transformer Model
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = TFGPT2LMHeadModel.from_pretrained("gpt2",
                                          pad_token_id=tokenizer.eos_token_id)

def generate_text(inp):
    # Encoding the starting point of the sentence we want to predict
    input_data = tokenizer.encode(inp,
                                  return_tensors='tf')
    # Generating Output String
    output = model.generate(
        input_data,
        do_sample=True,
        max_length=50,
        top_k=30
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)

gr.Interface(generate_text,
             "textbox",
             gr.Textbox()).launch(share=True) #, debug=True Use in Colab

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


TypeError: 'builtins.safe_open' object is not iterable

In [4]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# Load the data
df = pd.read_csv('/content/sample_data/california_housing_train.csv')

# Separate features (X) and target variable (y)
X = df.drop('median_house_value', axis=1)
y = df['median_house_value']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Create and train the linear regression model
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# Make predictions
y_pred = model.predict(X_test_scaled)

# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print('Model Performance:')
print(f'Root Mean Squared Error: ${rmse:.2f}')
print(f'R-squared Score: {r2:.3f}')

# Print feature importance
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_
})
print('\nFeature Importance:')
print(feature_importance.sort_values(by='Coefficient', key=abs, ascending=False))

Model Performance:
Root Mean Squared Error: $68078.33
R-squared Score: 0.664

Feature Importance:
              Feature   Coefficient
1            latitude -91983.102014
0           longitude -87098.433836
7       median_income  76754.043069
4      total_bedrooms  47893.657219
5          population -41374.290581
3         total_rooms -19175.064162
6          households  17328.745023
2  housing_median_age  14256.837611
